# Final Devel Benchmark Analysis
This notebook aggregates and analyzes all tracked provider/configuration benchmarks in `results_devel/provider_sweep.csv`, each run sequentially in isolated Slurm jobs with accurate whole-job cgroup memory tracking.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

# Load provider_sweep.csv
df = pd.read_csv('results_devel/provider_sweep.csv')

# Numeric conversions
df['cold_time_s'] = pd.to_numeric(df['cold_time_s'], errors='coerce')
df['warm_time_s'] = pd.to_numeric(df['warm_time_s'], errors='coerce')
df['solver_rss_mb'] = pd.to_numeric(df['solver_rss_mb'], errors='coerce')
df['job_mem_mb'] = pd.to_numeric(df['job_mem_mb'], errors='coerce')

# Feature engineering
def determine_plot_group(row):
    provider = str(row['provider'])
    if provider.startswith('PHYDLL_CPP'):
        return 'PHYDLL C++'
    if provider.startswith('PHYDLL_PY'):
        return 'PHYDLL Python'
    if 'BATCH' in provider:
        return 'BATCHED'
    if 'AIX' in provider:
        return 'AIX'
    return 'SMARTSIM'

df['plot_group'] = df.apply(determine_plot_group, axis=1)

# Summary table
df_success = df[df['status'] == 'SUCCESS'].copy()
summary = df_success.groupby(['model', 'provider']).agg(
    runs=('status', 'count'),
    cold_median=('cold_time_s', 'median'),
    warm_median=('warm_time_s', 'median'),
    solver_rss_median=('solver_rss_mb', 'median'),
    job_mem_median=('job_mem_mb', 'median')
).reset_index()
print("Summary of Devel Benchmark Runs:")
display(summary.sort_values(['model', 'warm_median']))


## Warm Inference Time (Bar Plots)
Median warm execution time per provider configuration and model.

In [ ]:
palette = {
    "AIX": "#4C72B0",
    "SMARTSIM": "#55A868",
    "BATCHED": "#D55E00",
    "PHYDLL C++": "#8172B2",
    "PHYDLL Python": "#CC79A7",
}

models = sorted(df_success['model'].unique())
for model in models:
    data_m = df_success[df_success['model'] == model]
    order = sorted(data_m['provider'].unique())
    
    plt.figure(figsize=(16, 8))
    ax = sns.barplot(data=data_m, x='provider', y='warm_time_s', hue='plot_group', 
                     palette=palette, estimator=np.median, order=order, dodge=False, errorbar=None)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10)
    
    plt.title(f"Median Warm Inference Time for Model: {model}", fontsize=18)
    plt.ylabel("Warm Time (s)", fontsize=14)
    plt.xlabel("Provider / Configuration", fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.legend(title="Provider Group")
    plt.tight_layout()
    plt.show()


## Whole-Job Memory Usage (cgroup Peak, MB)
Accurate peak memory usage recorded from Slurm cgroup (`memory.max_usage_in_bytes`) per provider job.

In [ ]:
for model in models:
    data_m = df_success[df_success['model'] == model]
    order = sorted(data_m['provider'].unique())
    
    plt.figure(figsize=(16, 8))
    ax = sns.barplot(data=data_m, x='provider', y='job_mem_mb', hue='plot_group', 
                     palette=palette, estimator=np.median, order=order, dodge=False, errorbar=None)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.0f', padding=3, fontsize=10)
    
    plt.title(f"Whole-Job cgroup Peak Memory for Model: {model}", fontsize=18)
    plt.ylabel("Peak Memory (MB)", fontsize=14)
    plt.xlabel("Provider / Configuration", fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.legend(title="Provider Group")
    plt.tight_layout()
    plt.show()
